# U2-L2：传感器速览

**课程目标**

1. 能说明 K1 上相机、IMU、里程计、关节、电池五类数据各自测的是什么，以及它们对应的 ROS2 消息类型。
2. 能比较「快照式取数」与「订阅式取数」两种读取方式，说明各自适合什么场合。
3. 能使用 SDK 的快照接口采集图像尺寸、姿态角、位置、电量，并读出这些数字的含义。
4. 能说明 `subscribe_image()` 与 `ros2 topic echo` 的关系：两者都是订阅者，区别只在于入口是 Python 还是命令行。


## 课程大纲映射（审核用）

| 轻任务 | 对应 cell | 核心概念 | 轻量概念 |
| --- | --- | --- | --- |
| ② 传感器速览：依次采集并打印图像尺寸、RPY、位置、电量% | 实验二 2.1、实验三 3.1–3.4 | 消息类型（`sensor_msgs/Image`、`sensor_msgs/Imu`、`nav_msgs/Odometry`） | — |
| `subscribe_image()` 本质是创建 ROS2 订阅者 | 实验二 2.2 | C04 订阅 | E02 回调 |

> 本映射块用于大纲对齐审核；导出 PDF 版本时移除此块。


## 一、传感器的数据也在话题上

上一课用 `ros2 topic list` 看到系统里有一堆话题，用 `hz` 测到图像话题一秒能出几十条消息。

### 1.1 五类数据，五种消息类型

| 传感器 | 测的是什么 | 话题上的消息类型 | SDK 里读到的对象 |
| --- | --- | --- | --- |
| 相机 | 前方的画面，一秒约 30 帧 | `sensor_msgs/msg/Image` | `Image` |
| IMU | 自身的加速度、转动快慢 | `sensor_msgs/msg/Imu` | `IMUState` |
| 里程计 | 走了多远、朝哪个方向 | `nav_msgs/msg/Odometry` | `OdomState` |
| 关节 | 每个关节的角度、速度、力矩 | `sensor_msgs/msg/JointState` | `JointStates` |
| 电池 | 剩余电量、电压、充放电状态 | `sensor_msgs/msg/BatteryState` | `BatteryState` |

消息里的字段名，和 SDK 对象里的属性名是同一套：
- 命令行里用 `ros2 topic echo` 看到的 `width`、`height`、`encoding`；
- 在 Python 里就是 `img.width`、`img.height`、`img.encoding`。

### 1.2 两种取数方式：快照与订阅

**快照式**：调用一次 `get_image()`、`get_imu()` 这类接口，拿回当前最新的一帧，调用完就结束。像举起手机拍一张照片，适合「我现在看一眼是多少」。

**订阅式**：调用一次 `subscribe_image(回调)` 把函数注册上去，之后每来一帧，SDK 自动帮你跑一次这个函数，直到你退订。像装了监控，画面自己推过来，适合「我要一直盯着，来一帧处理一帧」。

收到消息时自动跑的那段函数叫回调。`subscribe_image()` 要传进去的，就是这样一段函数。


## 二、连接 K1，读相机

**实验名称**：传感器速览

**实验目标**：采集 K1 相机的一帧图像，读出它的尺寸与像素编码；订阅相机两秒，数出一共收到多少帧；再依次读出 IMU 的加速度与角速度、里程计的位置与朝向、关节的角度、电池的剩余电量。

⚠️ **安全**：本课全程只读取传感器数据，**不发送任何运动 / 动作指令**，因此不需要让机器人进入 walk 模式。机器人放在地面，保持坐姿或趴姿即可；实验过程中无需搬动机器人。

**前置准备**：K1 已开机并与电脑建立连接（见 U1-L1）；已完成 U2-L1，会用 `ros2 topic list` 查看话题。

### 2.1 连接机器人


In [ ]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("已连接 K1")

### 2.2 抓一帧图像

`get_image()` 调用一次，拿回相机此刻最新的一帧。把 `img` 单独写在单元格最后一行，Jupyter 会把它显示成图片。


In [ ]:
img = robot.get_image()  # 抓一帧彩色图

print("尺寸：", img.width, "×", img.height)
print("编码：", img.encoding)
print("时间戳：", round(img.header.timestamp, 3))

img  # 写在最后一行，Jupyter 直接把这帧图显示出来

图像的关键信息：
- `width` 和 `height` 是画面的宽高像素数。
- `encoding` 说明像素怎么排列，这个字段在处理图像时要用，通道顺序弄反了，颜色会出错。
  - `rgb8` 表示每个像素按红、绿、蓝三个通道各占 1 字节存放，
  - `bgr8` 则是蓝、绿、红的顺序。


### 2.3 深度图（选做）

同一个函数换一个参数，拿到的就是深度图——它记录的不是颜色，而是每个像素离相机有多远。


In [ ]:
try:
    depth = robot.get_image(img_type="depth")
    print("尺寸：", depth.width, "×", depth.height)
    print("编码：", depth.encoding)
except Exception as e:
    print("这次没取到深度图：", e)
depth  # 写在最后一行，Jupyter 直接把这帧图显示出来

### 2.4 订阅相机两秒

下面注册一个回调，让它跑两秒，数一数这两秒里来了多少帧。


In [ ]:
import time

frames = []


def on_frame(img):
    """每收到一帧就跑一次，这里只记下它自带的时间戳。"""
    frames.append(img.header.timestamp)


sub = robot.subscribe_image(on_frame)    # 注册回调，本质是建一个订阅者
print("已订阅相机，等 2 秒……")
time.sleep(2)
sub.unsubscribe()                        # 用完退订，不再接收

print("2 秒内收到", len(frames), "帧")

`subscribe_image()` 是在相机的图像话题上建一个订阅者。

> 上一课的 `ros2 topic echo /boostercamera/head/rgb` 也是订阅，只不过它把消息打印在终端里；这里换成了你自己写的函数去订阅处理。

退订别漏：不退订，回调会一直跑，占着通道也占着算力。

把这里数出来的帧数除以 2，和上一课 `ros2 topic hz` 测到的帧率比一比，两个数字应该差不多。


## 三、IMU、里程计、关节与电池

<font color="red">下面继续只读传感器数据，不发送运动 / 动作指令，机器人不需要进入 walk 模式。</font> 实验过程中如有必要，保持现有姿势适当拖动机器人即可。

### 3.1 IMU：加速度与角速度

**（1）IMU是什么**

IMU, Inertial Measurement Unit（惯性测量单元），是 K1 胸腔内部集成的传感器模块，类比人类内耳中的前庭系统 —— 人闭眼能感知身体倾斜，靠的是前庭；K1 能感知自身倾斜，靠的是 IMU。

K1 搭载的是一枚 6 轴 IMU，“6 轴” 指的是 IMU 能同时感知 6 个物理量，它由两个核心元件组成：

| 元件　　　　　　　　　　　| 测量什么　　　　　　　　　　　　　| 类比　　　　　　　　　　　　　　|
| ---------------------------| -----------------------------------| ---------------------------------|
| 加速度计（Accelerometer） | 三个轴(x,y,z)上的加速度（含重力） | 感受颠簸、倾斜的 "重力方向感知" |
| 陀螺仪（Gyroscope）　　　 | 绕三个轴(x,y,z)的旋转角速度　　　 | 感受转弯、翻滚的 "旋转感知"　　 |


部分 IMU 还含第三种元件 —— 磁力计（含地磁方向），称为 9 轴 IMU；K1 使用的是不含磁力计的 6 轴 IMU， yaw 角依赖算法获得。

**（2）IMU 的功能与作用**

① 动态平衡控制（最核心）

K1 是双足人形机器人，重心高、支撑面小，**站立和行走全靠 IMU 实时反馈维持平衡：**
- 姿态反馈：IMU 每秒多次测量当前俯仰角（pitch）和侧倾角（roll），控制系统据此判断机器人是否倾斜、是否要摔倒。
- 平衡算法输入：K1 的站立稳定控制器（WBC）以 IMU 数据为输入，动态调整各关节扭矩，保持机体直立。
- 跌倒检测：倾斜超过安全阈值时，触发保护模式（PROTECT），防止摔坏关节。

>类比：想象闭眼单脚站立 —— 你之所以能坚持不倒，是因为内耳前庭在不断告诉大脑 "你现在往左偏了，往右纠正"。K1 的 IMU 就是机器人的 "内耳前庭"，控制系统就是它的 "大脑"。

② 运动感知与航迹推算

IMU 数据配合里程计（Odom），可以推算机器人在空间中的位置变化，用于：
- 路径规划与导航
- 移动中的姿态补偿
- 仿真环境中的运动状态监控

③ 传感器校准与诊断

通过读取 IMU 数据，可以判断机器人是否处于异常姿态（如被抬起、倾斜），以及各关节运动对整体姿态的影响。

**（3）获取IMU数据**

应用 `get_imu()` 可以获取IMU的数据。

In [ ]:
imu = robot.get_imu()

print("获取IMU的返回值：", imu)

`get_imu()` 的返回值是 `IMUState` 类型，包含如下信息：
| 属性　　　　　　　　　| 类型　　　　　　　　　 | 说明　　　　　　　　　　　　　　　　　　　　　　　　　　　　 |
| -----------------------| ------------------------| --------------------------------------------------------------|
| `header`　　　　　　　| `Header`　　　　　　　 | IMU元数据头　　　　　　　　　　　　　　　　　　　　　　　　　|
| `linear_acceleration` | `np.ndarray`　　　　　 | 线加速度 [x, y, z]，单位 m/s^2　　　　　　　　　　　　　　　 |
| `angular_velocity`　　| `np.ndarray`　　　　　 | 角速度 [x, y, z]，单位 rad/s　　　　　　　　　　　　　　　　 |
| `orientation`　　　　 | `np.ndarray` or `None` | 四元数 [x, y, z, w]　　　　　　　　　　　　　　　　　　　　　|
| `timestamp`　　　　　 | `float`　　　　　　　　| 秒级时间戳　　　　　　　　　　　　　　　　　　　　　　　　　 |
| `frame_id`　　　　　　| `str`　　　　　　　　　| 坐标系 ID　　　　　　　　　　　　　　　　　　　　　　　　　　|
| `rpy`　　　　　　　　 | `np.ndarray` or `None` | 欧拉角 [roll, pitch, yaw]，单位 rad；由 orientation 计算得到 |

获取`frame_id`,`linear_acceleration`,`angular_velocity`,`timestamp`查看。

In [ ]:
print("坐标系：", imu.frame_id)
print("线加速度 (m/s²)：", imu.linear_acceleration)
print("角速度 (rad/s)：", imu.angular_velocity)
print("时间戳：", round(imu.timestamp, 3))

机器人静止不动时，加速度里有一个轴的读数接近 9.8，那是重力。把它轻轻抬起、倾斜一下，三个轴的读数会重新分配；晃动的时候，角速度里会出现明显的数字。

`frame_id` 标出这组数据是相对哪个坐标系量出来的，读数据时先看它，才知道 x、y、z 分别朝哪。


### 3.2 里程计：位置与朝向

（1）什么是里程计

里程计（Odometry）是机器人用来估计自身位置和朝向的一类传感数据。它的核心原理是：通过读取关节编码器得到机器人实时的线速度和角速度，再通过特定算法就可以推算出机器人相对于起始点走了多远、转了多少度，从而得到当前在平面上的位姿（位置 + 朝向）。

对于 K1 这类足式人形机器人，里程计数据是通过腿部关节编码器的数据计算得到的，而不是来自 GPS 或视觉 —— 它是一种相对定位手段：以 "上电 / 重置时刻的位置" 为原点，累计推算后续位移。也正因为是*累计*，里程计会随时间和距离产生漂移（走得越远，误差越大），这是足式机器人的共性问题。

（2）功能与作用

里程计在 K1 开发中主要有以下用途：
- 实时位姿反馈：让程序知道机器人 "现在在哪、面朝哪个方向"，是所有移动类任务的基础输入。
- 闭环运动控制：例如让机器人精确走直线、走到指定坐标点 —— 程序不断读取里程计的 pose_2d，计算与目标的方向误差，动态修正角速度，从而避免开环控制下越走越偏。
- 导航与定位基础：里程计是路径规划、地图构建（Mapping）、定位（Localization）等上层功能的底层数据来源。
- 可重置原点：通过 reset_odom() 可以将当前位置设为新的坐标原点，方便在任务开始时统一坐标系。知识库

**（3）获取里程计数据**

应用 `get_odom()` 可以获取里程计的数据，返回的是机器人相对起点走了多远、朝哪个方向。

In [ ]:
odom = robot.get_odom()

print("获取的返回值：", odom)

`get_odom()` 的返回值是 `OdomState` 类型，包含如下信息：
| 属性　　　　　　　 | 类型　　　　　　　　　 | 说明　　　　　　　　　　　　　　　　　　　　　　　　　　　　 |
| --------------------| ------------------------| --------------------------------------------------------------|
| `header`　　　　　 | `Header`　　　　　　　 | 里程计元数据头　　　　　　　　　　　　　　　　　　　　　　　 |
| `position`　　　　 | `np.ndarray`　　　　　 | 位置 [x, y, z]，单位 m　　　　　　　　　　　　　　　　　　　 |
| `orientation`　　　| `np.ndarray`　　　　　 | 四元数 [x, y, z, w]　　　　　　　　　　　　　　　　　　　　　|
| `linear_velocity`　| `np.ndarray` or `None` | 线速度 [x, y, z]，单位 m/s　　　　　　　　　　　　　　　　　 |
| `angular_velocity` | `np.ndarray` or `None` | 角速度 [x, y, z]，单位 rad/s　　　　　　　　　　　　　　　　 |
| `timestamp`　　　　| `float`　　　　　　　　| 秒级时间戳　　　　　　　　　　　　　　　　　　　　　　　　　 |
| `frame_id`　　　　 | `str`　　　　　　　　　| 参考坐标系　　　　　　　　　　　　　　　　　　　　　　　　　 |
| `child_frame_id`　 | `str`　　　　　　　　　| 子坐标系　　　　　　　　　　　　　　　　　　　　　　　　　　 |
| `rpy`　　　　　　　| `np.ndarray`　　　　　 | 欧拉角 [roll, pitch, yaw]，单位 rad；由 orientation 计算得到 |
| `pose_2d`　　　　　| `np.ndarray`　　　　　 | 平面位姿 [x, y, yaw]　　　　　　　　　　　　　　　　　　　　 |

获取`position`,`rpy`,`pose_2d`查看。
- `position` 是 x、y、z 三个方向的位移，单位是米；
- `rpy` 把姿态换算成横滚（roll）、俯仰（pitch）、航向（yaw）三个角度，单位是弧度，下面通过数学变换换成度来读。

In [ ]:
import math

roll, pitch, yaw = odom.rpy
print("位置 x, y (m)：", round(odom.position[0], 3), round(odom.position[1], 3))
print("姿态 roll, pitch, yaw (度)：",
      round(math.degrees(roll), 1),
      round(math.degrees(pitch), 1),
      round(math.degrees(yaw), 1))
print("二维位姿 [x, y, yaw]：", odom.pose_2d)

将机器人趴或坐姿向前拖动一些，再次运行程序，会发现两次输出有些微不同。

In [ ]:
odom = robot.get_odom()
roll, pitch, yaw = odom.rpy

print("位置 x, y (m)：", round(odom.position[0], 3), round(odom.position[1], 3))
print("姿态 roll, pitch, yaw (度)：",
      round(math.degrees(roll), 1),
      round(math.degrees(pitch), 1),
      round(math.degrees(yaw), 1))
print("二维位姿 [x, y, yaw]：", odom.pose_2d)

`pose_2d` 把最常用的三个数打包在一起：x、y 和航向角 yaw。机器人在平面上走，判断「走到哪了、朝哪边」，用这三个数就够。


### 3.3 关节状态

`get_joint_states()` 一次把所有关节的当前状态都取回来。`len(js)` 是关节总数，`js.joints` 是逐个关节的列表。

提示：
- 机器人出去趴姿或坐姿，位于 `DAMP` 模式，可以轻微转动头部或任意关节，观察输出的不同。
- 前六个关节分别是：
    - 头部旋转`AAHead_Yaw`
    - 头部俯仰`Head_Pitch`
    - 左肩前摆`ALeft_Shoulder_Pitch`
    - 左键侧摆`Left_Shoulder_Roll`
    - 左臂旋转`Left_Elbow_Pitch`
    - 左肘`Left_Elbow_Yaw`

In [ ]:
js = robot.get_joint_states()

print("关节数量：", len(js))
print("时间戳：", round(js.timestamp, 3))
print()

for j in js.joints[:6]:          # 关节太多，先看前 6 个
    pos = "—" if j.position is None else round(j.position, 3)
    print(f"{j.name:<16} 角度 {pos}")

每个关节有 `name`、`position`、`velocity`、`effort` 四项：位置（弧度）、速度（弧度每秒）、力矩（牛·米）。

机器人摆的姿势不同，同一个关节读到的角度也不一样。需要单独查某一个关节时，用 `js.get_joint("关节名")`。


### 3.4 电池

`percentage` 是剩余电量百分比，`voltage` 是电压，`power_supply_status` 说明当前在充电还是放电。


In [ ]:
bat = robot.get_battery()
print("电量：", round(bat.percentage, 1), "%")
print("电压：", bat.voltage, "V")
print("状态：", bat.power_supply_status)

这几个数在真实运维里天天要看：机器人上电之后，先确认电量够不够跑完这一场，再开始下任务。


### 3.5 数据一直在刷新

同一个接口连着调两次，中间隔一秒，看看拿到的还是不是同一份数据。


In [ ]:
t1 = robot.get_odom().timestamp
time.sleep(1)
t2 = robot.get_odom().timestamp

print("第一次采集：", round(t1, 3))
print("第二次采集：", round(t2, 3))
print("中间隔了：", round(t2 - t1, 3), "秒")

两次调用拿到的时间戳不一样，中间差了大约 1 秒。这些数字一直在被刷新，每次调用取回的都是最新那一帧——这就是上一课说的「话题上不停往外冒的数据」，只不过这一回是在 Python 里看到的。


## 四、观察与记录

| 观察项 | 你看到的 | 说明 |
| --- | --- | --- |
| 相机图像的尺寸 |  | 宽 × 高各是多少像素？ |
| 图像的 `encoding` |  | 是 `rgb8` 还是 `bgr8`？ |
| 2 秒订阅收到的帧数 |  | 折算成每秒帧数，和上一课 `hz` 测到的一致吗？ |
| IMU 静止时的加速度 |  | 哪个轴接近 9.8？ |
| 里程计的位置 x, y |  | 机器人没动，数字是多少？ |
| 里程计的 yaw |  | 航向角大约多少度？ |
| 关节总数 |  | K1 一共多少个关节？ |
| 电池电量 |  | 剩多少？在充电还是放电？ |
| 两次采集的时间戳差 |  | 数据是不是一直在刷新？ |

**想一想**

1. `subscribe_image()` 和上一课的 `ros2 topic echo`，从哪些地方能看出它们做的是同一件事？
2. 要写一个「电量低于 20% 就告警」的程序，用快照式还是订阅式？说说你的理由。
3. 上面五类数据里，哪些在机器人静止不动时也一直在变？哪些基本不动？


## 五、扩展阅读：上电自检都看什么

展厅和工厂里的机器人，每天开机都要做一遍上电自检。运维人员按顺序确认：相机有没有出图、IMU 读数正不正常、里程计是不是从零点起算、电量够不够跑完这一场、关节有没有卡死。哪一样不对就先处理哪一样，处理完才开始任务。

这五项里，相机、IMU、里程计、电量、关节各占一项，和本课读到的传感器数据一一对应。会读这些数，是机器人应用开发和运维岗位的日常基本功。
